# Topic 1: Cluster Configuration & Resource Management

> **Phase 7 → Week 12 → Topic 1**
>
> Prerequisites: Week 6-7 (Performance), Week 8-9 (ETL)

---

## Spark Memory Model

```
Executor JVM Heap (spark.executor.memory = 8g):
┌─────────────────────────────────────────────────────────┐
│  Reserved Memory (300 MB fixed — for Spark internals)   │
├─────────────────────────────────────────────────────────┤
│  Usable Memory = (8g - 300MB) × spark.memory.fraction   │
│               = 7.7g × 0.6 = 4.62g                     │
│                                                         │
│  ┌───────────────────────┬─────────────────────────┐   │
│  │  Execution Memory     │   Storage Memory         │   │
│  │  (shuffle, sort,      │   (cache/persist,        │   │
│  │   aggregations)       │    broadcast vars)       │   │
│  │  spark.memory.        │   spark.memory.          │   │
│  │  storageFraction=0.5  │   storageFraction=0.5    │   │
│  │  ← can borrow from →  │   ← can borrow from →   │   │
│  └───────────────────────┴─────────────────────────┘   │
├─────────────────────────────────────────────────────────┤
│  User Memory = 7.7g × (1 - 0.6) = 3.08g               │
│  (UDFs, Python objects, non-Spark data structures)      │
└─────────────────────────────────────────────────────────┘

Overhead: spark.executor.memoryOverhead = max(executor_memory × 0.1, 384MB)
  → JVM overhead, Python worker, NIO buffers
  → TOTAL memory reserved on node = executor.memory + memoryOverhead
```

---

## Executor Sizing Formula

```
Rule of thumb: 5 cores per executor
  → Avoids HDFS throughput issues (>5 cores/executor can bottleneck HDFS)
  → Enough parallelism per executor without too much JVM overhead

Given a cluster with N nodes, each with C cores and M GB RAM:

  Step 1: cores_per_executor = 5 (or fewer if C < 5)
  Step 2: executors_per_node = floor((C - 1) / 5)
             Leave 1 core for OS/YARN daemons
  Step 3: memory_per_executor = floor((M - 1) / executors_per_node)
             Leave 1 GB for OS
  Step 4: memoryOverhead = max(memory_per_executor × 0.1, 384 MB)
  Step 5: actual_memory_per_executor = memory_per_executor - memoryOverhead

Example: 10 nodes, 16 cores, 64 GB RAM each
  executors_per_node = floor((16-1)/5) = 3
  total_executors = 10 × 3 = 30 (leave 1 for driver)
  memory_per_executor = floor((64-1)/3) = 21 GB
  memoryOverhead = max(21 × 0.1, 0.384) = 2.1 GB
  spark.executor.memory = 21 - 2.1 ≈ 19 GB
  spark.executor.cores = 5
  spark.executor.instances = 29  (30 - 1 for driver)
```

---

## Interview Cheat Sheet

**Q: What is the difference between `spark.executor.memory` and `spark.executor.memoryOverhead`?**
> `spark.executor.memory` is the JVM heap allocated to each executor. It covers Spark's execution memory (shuffle, aggregations) and storage memory (caching). `memoryOverhead` is additional off-heap memory for JVM metadata, Python workers (PySpark), NIO buffers, and other non-JVM processes. The node must have at least `executor.memory + memoryOverhead` available. OOM from PySpark UDFs often means insufficient `memoryOverhead`, not `executor.memory`.

**Q: What is Dynamic Allocation?**
> Dynamic Allocation lets Spark request more executors when there is a backlog of pending tasks and release idle executors when they've been unused for a timeout. This saves cluster cost on shared clusters. Enable with `spark.dynamicAllocation.enabled=true`. Requires an external shuffle service (so executors can release without losing shuffle data) or `spark.dynamicAllocation.shuffleTracking.enabled=true` (Spark 3.x).

**Q: How do you choose `spark.sql.shuffle.partitions`?**
> Rule of thumb: 2-3× the number of total executor cores. For a 30-executor × 5-core cluster = 150 cores → 300-400 shuffle partitions. Too few: data skew, OOM (large partitions). Too many: overhead from scheduling thousands of tiny tasks. AQE (Adaptive Query Execution) auto-coalesces shuffle partitions in Spark 3.0+ — set it high (e.g., 1000) and let AQE reduce.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Week12 - Cluster Configuration") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

## Part 1: Reading and Setting Spark Configuration

In [ ]:
# Read current configuration
def show_config(keys):
    for key in keys:
        try:
            val = spark.conf.get(key)
        except Exception:
            val = "(not set)"
        print(f"  {key} = {val}")

print("Memory configuration:")
show_config([
    "spark.executor.memory",
    "spark.executor.memoryOverhead",
    "spark.driver.memory",
    "spark.memory.fraction",
    "spark.memory.storageFraction",
])

print("\nCores and parallelism:")
show_config([
    "spark.executor.cores",
    "spark.executor.instances",
    "spark.sql.shuffle.partitions",
    "spark.default.parallelism",
])

print("\nAdaptive Query Execution:")
show_config([
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
])

print("\nRuntime:")
sc = spark.sparkContext
print(f"  Master:              {sc.master}")
print(f"  App ID:              {sc.applicationId}")
print(f"  Default parallelism: {sc.defaultParallelism}")
print(f"  Python version:      {sc.pythonVer}")

## Part 2: Dynamic Configuration at Runtime

In [ ]:
# Some configs can be changed at runtime (session-level)
# Others must be set at SparkSession creation time

print("Before:")
print("  shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

# Change shuffle partitions for a specific query
spark.conf.set("spark.sql.shuffle.partitions", "16")

data = spark.range(1000).withColumn("key", F.col("id") % 10)
result = data.groupBy("key").count()

print("After set (16 shuffle partitions):")
print("  shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("  actual shuffle partitions in result:", result.rdd.getNumPartitions())

# Reset
spark.conf.set("spark.sql.shuffle.partitions", "8")

print("\nConfigs that CANNOT be changed at runtime (must be in SparkSession builder):")
print("  spark.executor.memory")
print("  spark.executor.cores")
print("  spark.executor.instances")
print("  spark.master")
print("\nConfigs that CAN be changed at runtime:")
print("  spark.sql.shuffle.partitions")
print("  spark.sql.adaptive.enabled")
print("  spark.sql.autoBroadcastJoinThreshold")
print("  spark.sql.codegen.wholeStage")

## Part 3: Dynamic Allocation

In [ ]:
print("""
Dynamic Allocation Configuration:
════════════════════════════════════════════════════════════════

spark-submit \\
  --conf spark.dynamicAllocation.enabled=true \\
  --conf spark.dynamicAllocation.minExecutors=2 \\
  --conf spark.dynamicAllocation.maxExecutors=50 \\
  --conf spark.dynamicAllocation.initialExecutors=5 \\
  --conf spark.dynamicAllocation.executorIdleTimeout=60s \\
  --conf spark.dynamicAllocation.schedulerBacklogTimeout=1s \\
  --conf spark.shuffle.service.enabled=true \\
  my_job.py

Alternatively (Spark 3.x, without external shuffle service):
  --conf spark.dynamicAllocation.shuffleTracking.enabled=true

How it works:
  Scale UP:   If tasks are pending for > schedulerBacklogTimeout → add executors
  Scale DOWN: If executor idle for > executorIdleTimeout → remove it
              But DON'T remove if it holds shuffle data (external shuffle svc needed)

When to use dynamic allocation:
  ✓ Shared clusters (multiple users/jobs competing for resources)
  ✓ Jobs with variable workload (some stages much larger than others)
  ✓ Cost optimization on cloud (pay only for what you use)

When NOT to use:
  ✗ Streaming jobs (need stable number of tasks for partition assignment)
  ✗ Jobs with strict SLA (executor startup time adds latency)
  ✗ When you've already sized perfectly (fixed allocation is simpler)
════════════════════════════════════════════════════════════════
""")

## Part 4: spark-submit Flags Reference

In [ ]:
print("""
spark-submit Full Reference:
════════════════════════════════════════════════════════════════

spark-submit \\
  # Cluster manager
  --master yarn                         # YARN, local[N], k8s://..., spark://host:7077
  --deploy-mode cluster                 # cluster (driver on cluster) or client (driver local)

  # Resources per executor
  --executor-memory 8g                  # JVM heap per executor
  --executor-cores 5                    # cores per executor
  --num-executors 20                    # fixed executor count (static allocation)

  # Driver resources
  --driver-memory 4g
  --driver-cores 2

  # Application config
  --name "MyApp"
  --conf spark.sql.shuffle.partitions=400
  --conf spark.executor.memoryOverhead=2g
  --conf spark.sql.adaptive.enabled=true
  --conf spark.dynamicAllocation.enabled=true
  --conf spark.dynamicAllocation.maxExecutors=50

  # Dependencies
  --py-files utils.zip                  # Python packages/modules
  --jars /path/to/connector.jar         # Java JARs (Delta, JDBC driver, etc.)
  --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0  # Maven coordinates
  --files /path/to/config.json          # files distributed to each executor

  # Queue (YARN)
  --queue data_engineering

  # Application entrypoint
  my_job.py
  --date 2024-01-15
  --env prod

Sizing cheat sheet:
  Local dev:        local[4], 2g executor memory, 4 shuffle partitions
  Small cluster:    8g exec, 5 cores, 200 shuffle partitions
  Large cluster:    16g exec, 5 cores, dynamic alloc, 1000 shuffle partitions + AQE
  Streaming:        4g exec, fixed allocation (no dynamic), 10 shuffle partitions
════════════════════════════════════════════════════════════════
""")

## Part 5: Diagnosing OOM Errors

In [ ]:
print("""
OOM Diagnosis Guide:
════════════════════════════════════════════════════════════════

Symptom: ExecutorLostFailure / Java heap space / Cannot allocate memory

Step 1: Which type of OOM?
  "Java heap space"        → executor JVM heap full
  "GC overhead limit"      → executor spending >98% time in GC
  "Cannot allocate memory" → OS-level, often memoryOverhead too low
  Python Worker OOM        → PySpark UDF Python process, increase memoryOverhead

Step 2: Check Spark UI
  Stages tab → look for "Spill (Memory)" and "Spill (Disk)"
    Spill = not enough execution memory → increase executor.memory or reduce partitions
  Storage tab → cached DataFrames eating memory?
    → unpersist() DataFrames you no longer need

Step 3: Fix options (in order of preference)
  a) Increase shuffle partitions → smaller partitions → less memory per task
     spark.conf.set("spark.sql.shuffle.partitions", "400")

  b) Fix data skew → some partitions are 100× larger than others
     Use salting (Week 6) or AQE skew join handling

  c) Increase executor memory (last resort — costly)
     --executor-memory 16g

  d) For Python OOM (UDFs):
     --conf spark.executor.memoryOverhead=4g

  e) Reduce parallelism if tasks are too small
     (too many tiny tasks → overhead > computation)

  f) Replace groupByKey with reduceByKey / agg  (reduces shuffle data size)

  g) Use broadcast join instead of shuffle join for small tables
     F.broadcast(small_df)
════════════════════════════════════════════════════════════════
""")

## Exercises

1. You have a cluster: 20 nodes, 32 cores each, 128 GB RAM each. Calculate the optimal `--executor-memory`, `--executor-cores`, and `--num-executors` values using the 5-cores-per-executor rule.
2. Your PySpark job with a custom UDF fails with `Cannot allocate memory`. Explain what memory pool is exhausted and what configuration you change.
3. A job with AQE enabled uses 200 shuffle partitions but produces only 8 non-empty partitions. What does AQE do automatically? What would happen without AQE?
4. When should you use Dynamic Allocation and when should you avoid it? Give one concrete example of each.
5. Your executor has `spark.executor.memory=8g` and `spark.memory.fraction=0.6`. How much memory is available for shuffle/aggregations (execution memory) when storage memory is empty?